In [2]:
# Import Required Libraries 
import io
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
#     def read_data(): 
#     file_path = input("Enter your dataset path ")
#     cleaned_path = file_path.strip('"\'')
#     safe_path = Path(cleaned_path)
#     data = pd.read_csv(safe_path)
#     return data
    

In [4]:
#read_data()

In [5]:
# import pandas as pd

# # Use the 'r' prefix before the string to handle Windows backslashes safely
# path = r"C:\Users\Mosta\Downloads\8_4_competition_results.csv"
# df = pd.read_csv(path)

In [6]:
def user_choice():
    print("\nHow would you like to calculate your ROP and check stock levels?")
    print("  1. Read from dataset and SQL database")
    print("  2. ROP calculator(Manual)")
    choice = input("\nEnter 1 or 2: ").strip()
    return choice
    

In [ ]:
def manual_rop_calculator():
        config = {
        "current_stock" : int(input("Enter current stock level: ")),
        "avg_sales" : int(input("Enter average sales per day: ")),
        "std_dev_sales" : int(input("Enter standard deviation of sales per day: ")),
        "lead_time" : int(input("Enter lead time in days: ")),
        }
        config["safety_stock"] = 1.65 * config ["std_dev_sales"] * np.sqrt(config["lead_time"]),
        config["rop"] = (config["avg_sales"] * config["lead_time"]) + config["safety_stock"]
        if config["current_stock"] <= config["rop"]:
            print(f"⚠️  REORDER ALERT")
            print(f"    Current stock : {config['current_stock']} units")
            print(f"    Reorder point : {config['rop']:.1f} units")
            print(f"    → Place a new order now")
        else:
            buffer = config["current_stock"] - config["rop"]
            print(f"✅  Stock is OK")
            print(f"    Current stock : {config['current_stock']} units")
            print(f"    Reorder point : {config['rop']:.1f} units")
            print(f"    → {buffer:.1f} units above reorder point") 

In [8]:
def read_data_automatically():
    # 1. Create a visual upload button
    uploader = widgets.FileUpload(
        accept=".csv",  # Only allow CSV files
        multiple=False,  # Single file only
    )
    display(uploader)

    # 2. Internal function to process the file once clicked
    def on_upload_change(change):
        if not uploader.value:
            return

        # Grab the file data directly from memory
        uploaded_file = uploader.value[0]
        file_content = uploaded_file["content"]

        # Read it directly into pandas (no paths needed!)
        global data
        data = pd.read_csv(io.BytesIO(file_content))
        print(f"Successfully loaded: {uploaded_file['name']}")

    # Watch for the user to select a file
    uploader.observe(on_upload_change, names="value")
    

# Run this cell to show the button
# read_data_automatically()

In [9]:
def find_column(df, user_input):
    matched = [col for col in df.columns if col.lower() == user_input.lower().strip()]
    if matched:
        return matched[0]   # returns the real column name as it exists in the df
    raise ValueError(f"Column '{user_input}' not found. Available: {list(df.columns)}")

In [10]:
def find_date_column(df):
    synonyms = ['date', 'datetime', 'timestamp', 'time', 'day', 'period']
    for col in df.columns:
        if col.lower().strip() in synonyms:
            return col
    return None  # no date column found

In [11]:
def get_product(data):
    # global product_column
    # global targted_sales
    # global sales_column 

    product_column = find_column(data, input("Enter product column name: "))
    product_name   = input("Enter the product name: ").strip()          # plain input, no find_column
    sales_column   = find_column(data, input("Enter sales column name: "))
    targeted_sales = data.loc[data[product_column] == product_name].copy()
    return targeted_sales, sales_column
#get_product()

In [12]:
# second function 
# I can see the count of duplicates or nulls and upon it decide it to take action or not in if condition could be like if .duplicated().sum() ! = 0 : .....
def data_sentizer(targeted_sales , sales_column ):
    
    def duplicates_sentizer(targeted_sales):
        #global targted_sales
        if targeted_sales.duplicated().sum() != 0 :  
           targeted_sales = df.drop_duplicates().copy()
        return targeted_sales
    
    def outliers_sentizer(df):
        q1 = pd.Series(sorted(df[sales_column])).quantile(0.25)
        q3 = pd.Series(sorted(df[sales_column])).quantile(0.75)
        IQR = q3-q1
        upper_fence = q3 + 1.5 * IQR
        lower_fence = max(0, q1 - 1.5 * IQR)
        targeted_sales.loc[:, sales_column] = targeted_sales[sales_column].clip(lower_fence, upper_fence) # Uses explicit row/column indexing
        return targeted_sales
    
    def nulls_sentizer(targeted_sales):   
        if targeted_sales[sales_column].isna().sum() != 0 : 
            targeted_sales.loc[:, sales_column] = targeted_sales[sales_column].fillna(targeted_sales[sales_column].mean())       
        return targeted_sales
    targeted_sales = duplicates_sentizer(targeted_sales)
    targeted_sales = nulls_sentizer(targeted_sales)
    targeted_sales = outliers_sentizer(targeted_sales)
    return targeted_sales


   

In [13]:
def ROP (targeted_sales, sales_column):
    global avg_sales
    global lead_time
    # global _max_lead_time
    # global _avg_lead_time
    lead_time = int(input("Please Enter the lead time of the product: "))
    # _max_lead_time = int(input("Please Enter the Maximum lead time of the product: "))
    # _avg_lead_time = int(input("Please Enter the Average lead time of the product: "))
    avg_sales = targeted_sales[sales_column].mean()
    std_dev_sales = targeted_sales[sales_column].std()
    # Service level Z-score (1.65 = 95% confidence against stockouts)
    z_score = 1.65 
    
    # Statistical Safety Stock formula: Z * Standard_Deviation * sqrt(Lead_Time)
    safety_stock = z_score * std_dev_sales * np.sqrt(lead_time)
    
    # Reorder Point
    rop = (avg_sales * lead_time) + safety_stock
    # safety_stock =(max(targted_sales[sales_column]) * _max_lead_time) - (avg_sales * _avg_lead_time)
    # rop = (avg_sales * lead_time) + safety_stock
    return rop

In [15]:
def stock_alert(rop, targeted_sales):
    print("\nHow would you like to provide the current stock level?")
    print("  1. Read from dataset (if your dataset has an inventory column)")
    print("  2. Enter manually")
    choice = input("\nEnter 1 or 2: ").strip()
    if choice == "1" :
        date_col = find_date_column(targeted_sales)
        if date_col:
            latest_row = targeted_sales.sort_values(date_col).iloc[-1]
        else:
            print("  ℹ️  No date column found — using last row as most recent")
            latest_row = targeted_sales.iloc[-1]
        inventory_col = find_column(data, input("Enter inventory/stock column name: "))
        current_stock = latest_row[inventory_col]
        print(f"  → Latest inventory level from dataset: {current_stock} units")
    else:
        current_stock = int(input("Enter current stock level: "))
    print()
    if current_stock <= rop:
        print(f"⚠️  REORDER ALERT")
        print(f"    Current stock : {current_stock} units")
        print(f"    Reorder point : {rop:.1f} units")
        print(f"    → Place a new order now")
    else:
        buffer = current_stock - rop
        print(f"✅  Stock is OK")
        print(f"    Current stock : {current_stock} units")
        print(f"    Reorder point : {rop:.1f} units")
        print(f"    → {buffer:.1f} units above reorder point") 

In [16]:
data = read_data_automatically()

FileUpload(value=(), accept='.csv', description='Upload')

In [17]:
data.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [ ]:
def main():
    
    choice = user_choice()
    
    # Calculate ROP and check stock levels from dataset and SQL database
    if choice == "1" :
        config = {
        # Product
        "product_col": input("Product column name: "),
        "product_id":  input("Product ID: "),
        "sales_col":   input("Sales column name: "),
        
        # Settings
        "lead_time":        int(input("Lead time (days): ")),
        "restock_days":     int(input("How many days between restocks: ")),
        "is_perishable":    input("Perishable product? (yes/no): "),
        "shelf_life":       int(input("Shelf life in days (or 0): ")),
        "review_period":    int(input("Recalculate ROP every how many days? (90 recommended): "))

        }
        targeted_sales, sales_column = get_product(data)
        
        targeted_sales = data_sentizer(targeted_sales, sales_column)
        
        rop = ROP(targeted_sales, sales_column)
        
        stock_alert(rop , targeted_sales)
    
    # Calculate ROP and check stock levels manually
    else:
        manual_rop_calculator()
    # get_product()
    # data_sentizer()
    

In [21]:
main()


How would you like to calculate your ROP and check stock levels?
  1. Read from dataset and SQL database
  2. ROP calculator(Manual)

How would you like to provide the current stock level?
  1. Read from dataset (if your dataset has an inventory column)
  2. Enter manually
  → Latest inventory level from dataset: 267 units

⚠️  REORDER ALERT
    Current stock : 267 units
    Reorder point : 632.7 units
    → Place a new order now
